# Seed Viggle diffusion model (21GB via Range halves)

The diffusion model exceeds the 20GB kernel disk, so it's downloaded
in two ~10.5GB halves via HTTP Range requests. Attaches the v1 dataset
(small files) and publishes everything together as v2.
Run AFTER seed_viggle_small.


In [ ]:
# === Download diffusion model in two halves ===
import subprocess, os

SEED = "/kaggle/working/seed"
os.makedirs(SEED, exist_ok=True)
URL = "https://huggingface.co/drbaph/Viggle-Animate-ComfyUI/resolve/main/diffusion_models/minimax_h3_ref2va_viggle_pruned_int8_convrot.safetensors"
TOTAL = 21033720080
HALF = 10516860040
P1 = os.path.join(SEED, "diffusion_models", "diffusion_part1.bin")
P2 = os.path.join(SEED, "diffusion_models", "diffusion_part2.bin")
os.makedirs(os.path.dirname(P1), exist_ok=True)

def dl_range(out, start, end):
    if os.path.exists(out) and os.path.getsize(out) == (end - start + 1):
        print(f"SKIP {os.path.basename(out)} (present)")
        return
    print(f"Downloading bytes {start}-{end} ({(end-start+1)/1e9:.1f} GB)...")
    subprocess.run(["curl", "-s", "--show-progress",
        "-H", f"Range: bytes={start}-{end}", URL, "-o", out], check=True)
    sz = os.path.getsize(out)
    assert sz == (end - start + 1), f"SIZE MISMATCH: {sz}"
    print(f"  OK ({sz/1e9:.2f} GB)")

dl_range(P1, 0, HALF - 1)
dl_range(P2, HALF, TOTAL - 1)
print(f"\nBoth halves done. Total: {(os.path.getsize(P1)+os.path.getsize(P2))/1e9:.1f} GB")


In [ ]:
# === Reassemble + verify, then publish ===
import subprocess, json, os, hashlib

SEED = "/kaggle/working/seed"
DATASET_ID = "marsfmmg/viggle-animate-weights"
P1 = os.path.join(SEED, "diffusion_models", "diffusion_part1.bin")
P2 = os.path.join(SEED, "diffusion_models", "diffusion_part2.bin")
FULL = os.path.join(SEED, "diffusion_models",
    "minimax_h3_ref2va_viggle_pruned_int8_convrot.safetensors")

# Reassemble
print("Reassembling diffusion model...")
with open(FULL, "wb") as out:
    for p in [P1, P2]:
        with open(p, "rb") as f:
            while chunk := f.read(64*1024*1024):
                out.write(chunk)
assert os.path.getsize(FULL) == 21033720080, "reassembly size wrong"
print(f"Reassembled OK ({os.path.getsize(FULL)/1e9:.1f} GB)")
# Free disk before publish
os.remove(P1); os.remove(P2)
print("Halves removed, full file kept.")

# Publish (v1 small files are already in the dataset; this adds diffusion)
# NOTE: `kaggle datasets version` replaces content — copy v1 files first.
print("Copying v1 small files from attached dataset...")
subprocess.run(["cp", "-r", "/kaggle/input/viggle-animate-weights/loras",
    os.path.join(SEED, "loras")], check=True)
subprocess.run(["cp", "-r", "/kaggle/input/viggle-animate-weights/text_cond",
    os.path.join(SEED, "text_cond")], check=True)
subprocess.run(["cp", "-r", "/kaggle/input/viggle-animate-weights/vae",
    os.path.join(SEED, "vae")], check=True)
print("v1 files copied.")

meta = {"title": "Viggle-Animate Model Weights", "id": DATASET_ID,
        "licenses": [{"name": "Apache-2.0"}]}
with open(os.path.join(SEED, "dataset-metadata.json"), "w") as f:
    json.dump(meta, f, indent=2)
print("Publishing v2 (25GB, this takes a while)...")
r = subprocess.run(["kaggle", "datasets", "version", "-p", SEED,
    "-m", "Add diffusion model 21GB (reassembled from Range halves)"],
    capture_output=True, text=True, timeout=7200)
print((r.stdout or "")[-300:])
assert r.returncode == 0, "Publish FAILED: " + (r.stderr or "")[-500:]
print("\nDone. All Viggle weights cached.")
